## Building a character tokenizer

Why a character tokenizer?

Our input is "10 + 5 * 3" and out put is "25". We have some known operators * / + -, and the digits 1234567890 positive and negative signs.
We can't use word token or semi-word token because a number 123 is different from 321 or 124 or 12 or 23. Semi-number don't make sense, and
mapping every number is not possible. Our embedding is discrete not continuos.

In [61]:
import torch

class TinyArithmeticCharTokenizer:
    def __init__(self, embedding_dim: int):
        self.all_chars = "0123456789+-*/=."
        self.special_tokens = [" ", "<pad>", "<eos>", "<unk>"]
        self.vocab_list = list(self.all_chars) + self.special_tokens
        self.char_to_int = {char: i for i, char in enumerate(self.vocab_list)}
        self.int_to_char = {i: char for i, char in enumerate(self.vocab_list)}
        self.vocab_size = len(self.vocab_list)
        self.embedding = torch.nn.Embedding(num_embeddings=self.vocab_size, embedding_dim=embedding_dim)

    def encode(self, input_string: str) -> list[int]:
        return [self.char_to_int.get(char, self.char_to_int["<unk>"]) for char in input_string]

    def decode(self, token_ids: list[int]) -> str:
        decoded_chars = []
        for token_id in token_ids:
            char = self.int_to_char.get(token_id)
            if char is not None and char not in ["<eos>", "<pad>", "<unk>", " "]:
                decoded_chars.append(char)
            elif char == "<eos>":
                break
        return "".join(decoded_chars)

    def token_ids_to_embeddings(self, token_ids: list[int]) -> torch.Tensor:
        token_tensor = torch.tensor(token_ids).long()
        embedded_tokens = self.embedding(token_tensor)
        return embedded_tokens

    def input_to_embeddings(self, input_string: str) -> torch.Tensor:
        input_string += " = "
        return self.token_ids_to_embeddings(self.encode(input_string))

    def output_to_embeddings(self, output_string: str) -> torch.Tensor:
        output_token_ids = self.encode(output_string)
        output_token_ids.append(self.char_to_int["<eos>"])
        return self.token_ids_to_embeddings(output_token_ids)

In [65]:
tokenizer = TinyArithmeticCharTokenizer(embedding_dim=20)

print(f"vocab size: {tokenizer.vocab_size}")
print(f"embedding layer shape: {tokenizer.embedding.weight.shape}")

sample_input_string = "100 + 5 * 3"
sample_output_string = "25"

input_ids = tokenizer.encode(sample_input_string)
print(f"numerical IDs for input '{sample_input_string}': {input_ids}")

simulated_output_ids = tokenizer.encode(sample_output_string)
simulated_output_ids.append(tokenizer.char_to_int["<eos>"]) # model would typically output <eos>
print(f"\nsimulated output IDs: {simulated_output_ids}")
decoded_output = tokenizer.decode(simulated_output_ids)
print(f"decoded output: '{decoded_output}'")

vocab size: 20
embedding layer shape: torch.Size([20, 20])
numerical IDs for input '100 + 5 * 3': [1, 0, 0, 16, 10, 16, 5, 16, 12, 16, 3]

simulated output IDs: [2, 5, 18]
decoded output: '25'
